In [104]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [105]:
print("Length of text:", len(text), "\n")
print(text[:1000])

Length of text: 1115394 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, no

In [87]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)
print("".join(chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [74]:
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda x: ''.join(itos[i] for i in x)

In [88]:
decode(encode("Momo"))

'Momo'

In [93]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [94]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [95]:
torch.manual_seed(1337)
block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

xb, yb = get_batch("train")

In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, ix, targets=None):
        """
        ix : (B, T)
        """
        
        logits = self.token_embedding_table(ix) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, _ = logits.shape
            logits = logits.view(B*T, -1) # (B*T, C)
            targets = targets.view(-1) # (B*C)
            
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for i in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # (B, C): Only take the rows of the end tokens in each batch
            probs = F.softmax(logits, dim=-1)
            
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx
            

model = BigramLanguageModel(vocab_size)  
out, loss = model(xb, yb)

print(out.shape)
print(loss.item())

torch.Size([32, 65])
4.858022212982178


Initially all the values in the embedding matrix should be similar, approxiately 1/{vocab_size} . So the initial loss we should be expecting is -ln(1/65) = 4.174 

In [77]:
gen = model.generate(torch.randint(1, 65, (1, 1)), 100)
for b in gen:
    print("Generated text:\n", decode(b.tolist()))

Generated text:
 BSPy?,S.s,SpdDKmVjKs,lHQN.SaoCKlol3MAUtV&ulFZT'wFM?rsBg-bTsEwcxk?uQJjSVhnYQQcMOdDnqx?VyZvW!uL&yhRTb&N


What I got wrong here initially: I was thinking of maintaining the context size for this model and how it handles arbirtary long sentenses. But then I realised it is a bigram model and the context length by design is always 1. 

In [81]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [101]:
batch_size = 32
for steps in range(10000):

    xb, yb = get_batch("train")

    #evaluate
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.4309682846069336


In [102]:
gen = model.generate(torch.randint(1, 65, (1, 1)), 100)
for b in gen:
    print("Generated text:\n", decode(b.tolist()))

Generated text:
 wn f alis akis.
Th. h'ltondond a tot h ghy edisachoungoth maraw Ro,
TURONoulind s eemustin out athabe


## Averaging past context

In [121]:
B, T, C = 4, 8, 2

wei = torch.zeros((T, T))
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [123]:
x = torch.randn(B, T, C)
out = wei @ x # (T, T) @ (B, T, C) ----> (B, T, C)
print(x[0])
out[0]

tensor([[-0.2497,  0.1703],
        [ 1.6418, -1.0015],
        [ 1.7709, -0.5988],
        [ 0.5853,  1.4419],
        [ 0.3727,  0.3305],
        [ 0.2057,  0.6936],
        [ 0.2129,  1.6120],
        [-0.0623, -1.0430]])


tensor([[-0.2497,  0.1703],
        [ 0.6961, -0.4156],
        [ 1.0544, -0.4767],
        [ 0.9371,  0.0030],
        [ 0.8242,  0.0685],
        [ 0.7211,  0.1727],
        [ 0.6485,  0.3783],
        [ 0.5597,  0.2006]])

## Self attention

In [181]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C )

head_size = 16
query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x) # (B, T, h)
k = key(x) # (B, T, h)

# We need to previous weight matrix of all zeros with some similarity function wrt the q and k vectors
wei = q @ k.transpose(-2, -1) # (B, T, T)
# Now the averaging weights are determined by the data itself, so it is different for each batch

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

If we weighted sum all the previous contexts in the T-th token, we can then predict the next token only given the T-th token efficiently. So our most important task to find out the weights(masked) for the averaging operation. Ask yourself, which weights should be higher for average upto the T-th token, where the T-th token gets the averaged value ? Probably the ones the T-th token find more useful for prediction. So it spits out a query vector for all the tokens upto itelf to release their key vector for it to calculate a attention score, to be then softmaxed and and calculate the weights. 

To calculate the weighted avg itself, each vector has a different set of vectors called values, onto which the avg is calculated

Two type of broad distinction can be made in the attention mechanism, between self attention and cross-attention and between encoder and decoder attention module.

To understand the difference between them, take a step back and think about how general the attention mechanism itself is: you give it s 'set' of vectors, some of the vectors are directed to some of the other vectors or itself. This forms a directed graph, which represents a single batch of attention. The set of all the vectors a each vector is pointed to by participates in calculating the attentyion score for this vector, i.e. when a vectors spits out a query, the keys  and values are given by only the set of vectors it is being pointed to by. It does not need you to be autoregressive(i.e. causal) or fully connected or anything like that.

With this view in mind, the difference between the mechanisms should fairly simple. In self attention each vector in the set gives a query and a subset of vectors in the same set gives their key and values for each query. In cross attention, you can divide the vectors into two subsets, one of which gives out query but the keys and the values is only given by the other subset of vectors and the other set of vectors is pointed to by none, so they dont give query vectors. So the edges go from only one set of vectors to the other and neither the other way nor to any vector in itself. 

The encoder and dedocer is most widely used but specific type of attention. The encoder is a [complete directed graph](https://en.wikipedia.org/wiki/Directed_graph#:~:text=Complete%20directed%20graphs%20are%20simple%20directed%20graphs%20where%20each%20pair%20of%20vertices%20is%20joined%20by%20a%20symmetric%20pair%20of%20directed%20arcs%20(it%20is%20equivalent%20to%20an%20undirected%20complete%20graph%20with%20the%20edges%20replaced%20by%20pairs%20of%20inverse%20arcs).%20It%20follows%20that%20a%20complete%20digraph%20is%20symmetric.), where each vector points to each of the vectors, including itself, so each one gives a query and all the vectors attend to it with keys and values. On the other hand decoder block is used for causal auto regressive tasks where the vectors are ordered and each vector is pointed by the all the vectors ordered upto itself.

Because or the exponential (peaky) nature of softmax, we used scaled dot product attention, other wise the softmax converges to one-hot vectors to the max value.